# Wildlife identification from a photo

Use Claude's vision capability via the Messages API to identify wildlife in a photograph and
return a structured analysis: subject detection, species identification, plausible look-alikes,
habitat cues, and a 1-4 confidence rating.

**Requirements:** an `ANTHROPIC_API_KEY` in your environment (or a `.env` file at the repo
root). See the repository README for setup.

In [ ]:
# Load env variables and create client
import os
import sys

from dotenv import load_dotenv
from anthropic import Anthropic

# Shared building blocks (model, prompt, image helpers) live in _wildlife.py so this
# notebook and the Streamlit app stay in sync. Make it importable whether the working
# directory is this folder or the repo root.
for _p in (".", "vision"):
    if os.path.isfile(os.path.join(_p, "_wildlife.py")) and _p not in sys.path:
        sys.path.insert(0, _p)

from _wildlife import MODEL, PROMPT, url_image_block, image_block

load_dotenv()

client = Anthropic()
model = MODEL

In [ ]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


# url_image_block and image_block are imported from _wildlife above.

## The image source

The Messages API can analyze an image supplied either as base64-encoded bytes from a local
file or fetched directly from a public URL. This notebook uses a URL so it stays
self-contained with no local file to manage.

The photo below is a red fox from the U.S. National Park Service (Cedar Breaks National
Monument). NPS photographs are works of the federal government and are in the public domain,
so they are safe to use in examples and teaching material.

Source: https://www.nps.gov/cebr/learn/nature/red-fox.htm

To run the same analysis on your own photo, swap `url_image_block(...)` for
`image_block("your_photo.jpg")` further down.

In [ ]:
# The analysis prompt is defined once in _wildlife.py (shared with the Streamlit app).
# Print it here so the notebook stays self-documenting.
print(PROMPT)

In [ ]:
# Feed the image into Claude
image_url = "https://www.nps.gov/cebr/learn/nature/images/RedFox_4.jpg"

messages = []
add_user_message(
    messages,
    [
        url_image_block(image_url),
        {"type": "text", "text": PROMPT},
    ],
)

# To use a local photo instead, comment out the block above and use:
# add_user_message(
#     messages,
#     [image_block("your_photo.jpg"), {"type": "text", "text": PROMPT}],
# )

response = chat(messages)
print(text_from_message(response))

## Where was it taken? Two different answers

Step 5 of the prompt asks Claude to **estimate** a region from visual cues alone — species
range, vegetation, terrain, light. That is inference from the pixels, never exact coordinates.

A photo can also carry the *precise* location in its **EXIF metadata** (the GPS tags a phone or
camera writes into the file). Claude never sees that metadata, so we read it ourselves with the
`extract_gps` helper. Two caveats make the empty result below the normal case:

- Claude fetches a URL image on its own, so to inspect metadata we download the bytes separately.
- Most images published on the web — including this NPS photo — have had their EXIF **stripped**,
  so there are no coordinates to find. Try it on a photo straight from your phone to see real
  coordinates come back.

In [ ]:
# Read precise GPS coordinates from the photo's EXIF metadata (when present)
from _wildlife import extract_gps, fetch_image_bytes

img_bytes = fetch_image_bytes(image_url)
gps = extract_gps(img_bytes) if img_bytes else None
print(gps or "No GPS metadata in this image (EXIF was likely stripped before publishing).")

# For a local photo from your phone, read its bytes directly instead:
# with open("your_photo.jpg", "rb") as f:
#     print(extract_gps(f.read()))